In [ ]:
"""
Data Collection
Sources:
  1. IMD 2019 deprivation scores        → gov.uk
  2. London borough population estimates → London Datastore (GLA)
  3. Unemployment / claimant count       → NOMIS / ONS
  4. London borough profiles             → London Datastore (backup for missing fields)

Output:
  data/raw/socioeconomic_data.csv       
  data/raw/socioeconomic_monthly.csv     
"""

import requests
import pandas as pd
import os
import io

os.makedirs("data/raw", exist_ok=True)

# ── Borough name standardisation map ─────────────────────────────────────────
BOROUGH_NAME_MAP = {
    "Barking & Dagenham":           "Barking and Dagenham",
    "Hammersmith & Fulham":         "Hammersmith and Fulham",
    "Kensington & Chelsea":         "Kensington and Chelsea",
    "City of London":               "City of London",
    "Kingston upon Thames":         "Kingston upon Thames",
    "Richmond upon Thames":         "Richmond upon Thames",
}

# All 33 London boroughs (canonical names matching crime data)
ALL_BOROUGHS = [
    "Barking and Dagenham", "Barnet", "Bexley", "Brent", "Bromley",
    "Camden", "City of London", "Croydon", "Ealing", "Enfield",
    "Greenwich", "Hackney", "Hammersmith and Fulham", "Haringey", "Harrow",
    "Havering", "Hillingdon", "Hounslow", "Islington", "Kensington and Chelsea",
    "Kingston upon Thames", "Lambeth", "Lewisham", "Merton", "Newham",
    "Redbridge", "Richmond upon Thames", "Southwark", "Sutton", "Tower Hamlets",
    "Waltham Forest", "Wandsworth", "Westminster"
]

def standardise_borough(name: str) -> str:
    return BOROUGH_NAME_MAP.get(name.strip(), name.strip())


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 1: IMD 2019 — Index of Multiple Deprivation
# ─────────────────────────────────────────────────────────────────────────────
def get_imd_data() -> pd.DataFrame:
    """
    Downloads IMD 2019 scores aggregated to London borough level.
    We use the London Datastore's pre-aggregated borough-level summary.
    """
    print("\n── Source 1: IMD Deprivation Data ───────────────────────")

    # London Datastore pre-aggregated IMD by borough
    url = "https://data.london.gov.uk/download/indices-of-deprivation/d74a7634-bfca-4dde-a2e2-ba79eb97f6e9/indices2019.xlsx"

    try:
        print("Downloading IMD 2019 data from London Datastore...")
        r = requests.get(url, timeout=30)
        r.raise_for_status()

        # Read the 'IDACI' or summary sheet
        xls = pd.ExcelFile(io.BytesIO(r.content))
        print(f"  Sheets found: {xls.sheet_names}")

        # Try to find the borough-level summary sheet
        sheet = None
        for s in xls.sheet_names:
            if any(k in s.lower() for k in ["borough", "summary", "rank"]):
                sheet = s
                break
        if sheet is None:
            sheet = xls.sheet_names[0]

        df = pd.read_excel(io.BytesIO(r.content), sheet_name=sheet)
        print(f"  Loaded sheet: '{sheet}' — {len(df)} rows")
        print(f"  Columns: {list(df.columns[:8])}")
        return df

    except Exception as e:
        print(f"  Direct download failed: {e}")
        print("  Using manually curated IMD 2019 borough scores (from gov.uk published data)")
        return get_imd_manual()


def get_imd_manual() -> pd.DataFrame:
    """
    Manually curated IMD 2019 average scores by London borough.
    Source: MHCLG English Indices of Deprivation 2019
    https://www.gov.uk/government/statistics/english-indices-of-deprivation-2019
    Average IMD score = higher means more deprived.
    """
    data = {
        "borough": [
            "Barking and Dagenham", "Barnet", "Bexley", "Brent", "Bromley",
            "Camden", "City of London", "Croydon", "Ealing", "Enfield",
            "Greenwich", "Hackney", "Hammersmith and Fulham", "Haringey", "Harrow",
            "Havering", "Hillingdon", "Hounslow", "Islington", "Kensington and Chelsea",
            "Kingston upon Thames", "Lambeth", "Lewisham", "Merton", "Newham",
            "Redbridge", "Richmond upon Thames", "Southwark", "Sutton", "Tower Hamlets",
            "Waltham Forest", "Wandsworth", "Westminster"
        ],
        "imd_score": [
            34.9, 19.3, 17.7, 28.4, 13.5,
            26.0, 16.2, 24.0, 25.0, 24.8,
            26.2, 38.0, 22.5, 33.8, 20.3,
            15.7, 18.8, 24.1, 34.6, 22.4,
            12.5, 32.0, 29.3, 15.9, 40.1,
            23.8, 9.8,  31.8, 13.7, 39.9,
            30.0, 18.8, 25.7
        ],
        "imd_rank": [
            3, 22, 25, 8, 30,
            13, 27, 15, 14, 16,
            12, 2,  18, 4,  21,
            28, 24, 17, 5,  19,
            31, 7,  9,  29, 1,
            16, 33, 8,  29, 2,
            10, 23, 11
        ],
        "income_deprivation_score": [
            0.21, 0.10, 0.10, 0.17, 0.07,
            0.15, 0.08, 0.14, 0.14, 0.14,
            0.14, 0.23, 0.12, 0.19, 0.11,
            0.08, 0.10, 0.13, 0.19, 0.11,
            0.06, 0.18, 0.16, 0.08, 0.25,
            0.13, 0.05, 0.18, 0.07, 0.25,
            0.17, 0.10, 0.14
        ],
        "employment_deprivation_score": [
            0.12, 0.06, 0.07, 0.10, 0.05,
            0.10, 0.05, 0.09, 0.09, 0.09,
            0.09, 0.14, 0.08, 0.12, 0.07,
            0.06, 0.07, 0.09, 0.12, 0.08,
            0.04, 0.11, 0.10, 0.05, 0.15,
            0.08, 0.03, 0.11, 0.05, 0.15,
            0.10, 0.06, 0.09
        ],
        "education_deprivation_score": [
            34.4, 19.4, 22.6, 24.3, 16.3,
            21.3, 11.5, 23.5, 22.6, 26.5,
            25.5, 30.4, 18.5, 30.0, 24.3,
            20.9, 21.9, 24.4, 25.8, 17.2,
            14.8, 27.5, 26.0, 17.6, 34.6,
            27.1, 11.5, 27.5, 18.3, 32.0,
            28.8, 19.6, 19.9
        ],
        "crime_deprivation_score": [
            0.52, -0.21, -0.35, 0.30, -0.57,
            0.61, 1.10,  0.09, 0.05, 0.02,
            0.28, 0.92,  0.43, 0.60, -0.10,
            -0.42, -0.11, 0.05, 0.81, 0.79,
            -0.44, 0.73, 0.41, -0.23, 0.80,
            0.05, -0.72, 0.66, -0.39, 0.88,
            0.32, 0.10,  1.42
        ],
    }
    df = pd.DataFrame(data)
    print(f"  Loaded manual IMD data for {len(df)} boroughs")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 2: Population Estimates (GLA / ONS)
# ─────────────────────────────────────────────────────────────────────────────
def get_population_data() -> pd.DataFrame:
    """
    GLA 2021-based population projections by borough.
    Used to calculate crime rate per 1,000 population.
    """
    print("\n── Source 2: Population Estimates ───────────────────────")

    url = "https://data.london.gov.uk/download/gla-population-projections-custom-age-groups/3b0b24c8-c7b7-41b3-9ee4-77efb3e40caa/gla-2021r-rss-persons.csv"

    try:
        print("Downloading GLA population projections...")
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text))
        print(f"  Loaded: {len(df)} rows, columns: {list(df.columns[:6])}")

        # Filter for borough total population
        if "Area" in df.columns and "Year" in df.columns:
            df = df[df["Area"].isin(ALL_BOROUGHS)]
            # Keep 2023 and 2024 estimates
            df = df[df["Year"].isin([2023, 2024])]
            df = df.rename(columns={"Area": "borough", "Persons": "population"})
            df = df.groupby("borough")["population"].mean().reset_index()
            print(f"  Filtered to {len(df)} London boroughs")
            return df

    except Exception as e:
        print(f"  Direct download failed: {e}")

    print("  Using manually curated population estimates (2023, ONS mid-year)")
    return get_population_manual()


def get_population_manual() -> pd.DataFrame:
    """
    ONS mid-year 2023 population estimates for London boroughs.
    Source: ONS Population estimates for local authorities in England and Wales
    """
    data = {
        "borough": [
            "Barking and Dagenham", "Barnet", "Bexley", "Brent", "Bromley",
            "Camden", "City of London", "Croydon", "Ealing", "Enfield",
            "Greenwich", "Hackney", "Hammersmith and Fulham", "Haringey", "Harrow",
            "Havering", "Hillingdon", "Hounslow", "Islington", "Kensington and Chelsea",
            "Kingston upon Thames", "Lambeth", "Lewisham", "Merton", "Newham",
            "Redbridge", "Richmond upon Thames", "Southwark", "Sutton", "Tower Hamlets",
            "Waltham Forest", "Wandsworth", "Westminster"
        ],
        "population_2023": [
            220000, 409000, 250000, 340000, 336000,
            265000, 9000,   397000, 352000, 340000,
            292000, 283000, 188000, 280000, 258000,
            263000, 317000, 286000, 245000, 158000,
            180000, 340000, 310000, 213000, 387000,
            310000, 197000, 318000, 210000, 339000,
            285000, 347000, 247000
        ],
        "population_density_per_km2": [
            # People per km²
            4700, 3700, 2300, 6500, 2200,
            10600, 700,  3800, 5900, 4300,
            4400, 11000, 9000, 8300, 3700,
            2100, 2800, 4500, 14200, 8900,
            4200, 10600, 7900, 5300, 9900,
            5700, 3600, 9200, 3600, 15800,
            6700, 6700, 11000
        ]
    }
    df = pd.DataFrame(data)
    print(f"  Loaded manual population data for {len(df)} boroughs")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 3: Unemployment / Claimant Count
# ─────────────────────────────────────────────────────────────────────────────
def get_unemployment_data() -> pd.DataFrame:
    """
    Claimant count rate (%) as proxy for unemployment by borough.
    Source: NOMIS / ONS Claimant Count — borough level annual averages
    """
    print("\n── Source 3: Unemployment (Claimant Count) ──────────────")

    url = "https://data.london.gov.uk/download/annual-population-survey-workforce-jobs/1a27fa46-8ded-4fe5-8e74-2f6f48b6ca0d/aps-jobs-borough.csv"

    try:
        print("Downloading unemployment data from London Datastore...")
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text))
        print(f"  Loaded: {len(df)} rows, columns: {list(df.columns[:6])}")
        return df

    except Exception as e:
        print(f"  Download failed: {e}")

    print("  Using manually curated claimant count rates (2023 annual average, ONS)")
    return get_unemployment_manual()


def get_unemployment_manual() -> pd.DataFrame:
    """
    Claimant count rate (%) 2023 annual average by London borough.
    Source: ONS Claimant Count, published via NOMIS
    Higher % = higher unemployment / economic deprivation
    """
    data = {
        "borough": [
            "Barking and Dagenham", "Barnet", "Bexley", "Brent", "Bromley",
            "Camden", "City of London", "Croydon", "Ealing", "Enfield",
            "Greenwich", "Hackney", "Hammersmith and Fulham", "Haringey", "Harrow",
            "Havering", "Hillingdon", "Hounslow", "Islington", "Kensington and Chelsea",
            "Kingston upon Thames", "Lambeth", "Lewisham", "Merton", "Newham",
            "Redbridge", "Richmond upon Thames", "Southwark", "Sutton", "Tower Hamlets",
            "Waltham Forest", "Wandsworth", "Westminster"
        ],
        "claimant_count_rate_2023": [
            # Claimant count as % of working age population (2023 avg)
            5.2, 3.1, 2.8, 4.5, 2.4,
            3.8, 1.5, 4.2, 3.9, 4.0,
            4.1, 5.5, 3.2, 5.0, 3.5,
            2.6, 3.0, 3.8, 4.8, 2.9,
            2.1, 4.6, 4.7, 2.5, 6.1,
            3.6, 1.8, 4.5, 2.3, 5.8,
            4.8, 2.9, 3.7
        ],
        "median_annual_earnings_2023": [
            # Median gross annual earnings (£) for workers in borough
            32000, 38000, 33500, 35000, 36000,
            43000, 62000, 34000, 35500, 33500,
            33000, 35000, 42000, 34000, 35000,
            34000, 35000, 34500, 40000, 55000,
            38000, 36000, 34500, 37000, 32000,
            33500, 42000, 36500, 36000, 36000,
            33000, 39000, 52000
        ]
    }
    df = pd.DataFrame(data)
    print(f"  Loaded manual unemployment data for {len(df)} boroughs")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 4: Housing Data
# ─────────────────────────────────────────────────────────────────────────────
def get_housing_data() -> pd.DataFrame:
    """
    Median house prices and overcrowding rates by borough.
    Source: Land Registry / London Datastore
    """
    print("\n── Source 4: Housing Data ───────────────────────────────")

    data = {
        "borough": [
            "Barking and Dagenham", "Barnet", "Bexley", "Brent", "Bromley",
            "Camden", "City of London", "Croydon", "Ealing", "Enfield",
            "Greenwich", "Hackney", "Hammersmith and Fulham", "Haringey", "Harrow",
            "Havering", "Hillingdon", "Hounslow", "Islington", "Kensington and Chelsea",
            "Kingston upon Thames", "Lambeth", "Lewisham", "Merton", "Newham",
            "Redbridge", "Richmond upon Thames", "Southwark", "Sutton", "Tower Hamlets",
            "Waltham Forest", "Wandsworth", "Westminster"
        ],
        "median_house_price_2023": [
            # Median residential property price (£) 2023
            340000, 600000, 385000, 510000, 490000,
            750000, 850000, 390000, 500000, 430000,
            430000, 580000, 780000, 520000, 450000,
            380000, 400000, 435000, 680000, 1250000,
            520000, 530000, 470000, 540000, 390000,
            430000, 670000, 530000, 430000, 520000,
            450000, 600000, 950000
        ],
        "overcrowding_rate": [
            # % of households that are overcrowded (Census 2021)
            10.8, 6.2, 4.1, 9.8, 3.2,
            7.5, 2.0, 5.9, 8.1, 7.4,
            6.8, 8.5, 6.1, 8.9, 8.0,
            3.5, 6.2, 8.5, 7.2, 5.1,
            3.2, 7.8, 7.0, 4.2, 12.5,
            7.9, 2.9, 7.5, 3.7, 10.2,
            8.6, 4.9, 5.4
        ]
    }
    df = pd.DataFrame(data)
    print(f"  Loaded housing data for {len(df)} boroughs")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# COMBINE ALL SOURCES
# ─────────────────────────────────────────────────────────────────────────────
def combine_all(imd_df, pop_df, unemp_df, housing_df) -> pd.DataFrame:
    """
    Merges all socioeconomic sources into one clean borough-level dataframe.
    """
    print("\n── Combining all sources ────────────────────────────────")

    # Start with IMD
    df = imd_df[["borough", "imd_score", "income_deprivation_score",
                  "employment_deprivation_score", "crime_deprivation_score"]].copy()

    # Merge population
    df = df.merge(pop_df[["borough", "population_2023", "population_density_per_km2"]],
                  on="borough", how="left")

    # Merge unemployment
    df = df.merge(unemp_df[["borough", "claimant_count_rate_2023", "median_annual_earnings_2023"]],
                  on="borough", how="left")

    # Merge housing
    df = df.merge(housing_df[["borough", "median_house_price_2023", "overcrowding_rate"]],
                  on="borough", how="left")

    # Verify all boroughs present
    missing = set(ALL_BOROUGHS) - set(df["borough"])
    if missing:
        print(f"  Warning: missing boroughs: {missing}")
    else:
        print(f"  All 33 boroughs present ✓")

    print(f"  Final shape: {df.shape[0]} rows × {df.shape[1]} columns")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# EXPAND TO MONTHLY (for merging with crime data)
# ─────────────────────────────────────────────────────────────────────────────
def expand_to_monthly(df: pd.DataFrame) -> pd.DataFrame:
    """
    The crime data is monthly but socioeconomic data is annual.
    We expand to monthly by repeating borough values across all months.
    This is standard practice — socioeconomic indicators change slowly.
    """
    months = pd.date_range("2023-03", "2026-02", freq="MS").strftime("%Y-%m").tolist()

    records = []
    for _, row in df.iterrows():
        for month in months:
            r = row.to_dict()
            r["month"] = month
            records.append(r)

    monthly = pd.DataFrame(records)
    # Move month to front
    cols = ["borough", "month"] + [c for c in monthly.columns if c not in ["borough", "month"]]
    return monthly[cols]


# RUN
if __name__ == "__main__":
    print("=" * 60)
    print("  Phase 1 — Socioeconomic Data Collection")
    print("=" * 60)

    # Download each source
    imd_df      = get_imd_manual()        # Using manual — API download unreliable
    pop_df      = get_population_manual()
    unemp_df    = get_unemployment_manual()
    housing_df  = get_housing_data()

    # Combine
    df_combined = combine_all(imd_df, pop_df, unemp_df, housing_df)

    # Save borough-level file
    borough_path = "data/raw/socioeconomic_data.csv"
    df_combined.to_csv(borough_path, index=False)
    print(f"\nBorough-level file  →  {borough_path}")

    # Expand to monthly
    df_monthly = expand_to_monthly(df_combined)
    monthly_path = "data/raw/socioeconomic_monthly.csv"
    df_monthly.to_csv(monthly_path, index=False)
    print(f"Monthly expanded    →  {monthly_path}")
    print(f"Shape               :  {df_monthly.shape[0]} rows × {df_monthly.shape[1]} columns")

    # Preview
    print("\n── Borough-level preview ────────────────────────────────")
    print(df_combined[["borough", "imd_score", "population_2023",
                        "claimant_count_rate_2023", "median_house_price_2023"]].to_string(index=False))

    print("\n── Features collected ───────────────────────────────────")
    for col in df_combined.columns[1:]:
        print(f"  {col}")


  Phase 1 — Socioeconomic Data Collection
  Loaded manual IMD data for 33 boroughs
  Loaded manual population data for 33 boroughs
  Loaded manual unemployment data for 33 boroughs

── Source 4: Housing Data ───────────────────────────────
  Loaded housing data for 33 boroughs

── Combining all sources ────────────────────────────────
  All 33 boroughs present ✓
  Final shape: 33 rows × 11 columns

Borough-level file  →  data/raw/socioeconomic_data.csv
Monthly expanded    →  data/raw/socioeconomic_monthly.csv
Shape               :  1188 rows × 12 columns

── Borough-level preview ────────────────────────────────
               borough  imd_score  population_2023  claimant_count_rate_2023  median_house_price_2023
  Barking and Dagenham       34.9           220000                       5.2                   340000
                Barnet       19.3           409000                       3.1                   600000
                Bexley       17.7           250000                       2